## Store Sales Forecasting Pipeline - Delhi Data Scientist Assignment
- **Author -** Shivam Chaudhary
- **Objective -** Data preprocessing, feature engineering, LSTM model training, and exporting artifacts for deployment.

In [7]:
# Import Libraries
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import pickle
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler

In [16]:
# 1. Load the original dataset
print("Loading raw store sales excel sheet...")
df_raw = pd.read_excel('Store Sales Hoi Assignment Data Scientist.xlsx', sheet_name='Sheet1')
df_raw.columns = df_raw.columns.astype(str)

Loading raw store sales excel sheet...


In [17]:
# Fix active/inactive store gaps using forward/backward fill strategy
# This ensures structural continuity for our sequential LSTM window
df_clean = df_raw.copy()
df_clean.iloc[:, 1:] = df_clean.iloc[:, 1:].ffill().bfill().fillna(0)

In [18]:
# 2. Custom Feature Engineering Function
def build_features(df):
    data = df.copy()
    
    # Base calendar features
    data['DayOfWeek'] = data['Date'].dt.dayofweek
    data['IsWeekend'] = data['DayOfWeek'].isin([5, 6]).astype(int)
    data['Month'] = data['Date'].dt.month
    
    # Cyclical transformations for monthly patterns
    data['Sin_Month'] = np.sin(2 * np.pi * data['Month'] / 12)
    data['Cos_Month'] = np.cos(2 * np.pi * data['Month'] / 12)
    
    # Key Indian national holidays affecting Delhi retail sales
    data['IsHoliday'] = data['Date'].apply(
        lambda x: 1 if (x.month == 1 and x.day == 26) or 
                       (x.month == 8 and x.day == 15) or 
                       (x.month == 10 and x.day == 2) or 
                       (x.month == 12 and x.day == 25) else 0
    )
    
    # Monthly historical average temperature proxy for Delhi weather impact
    delhi_temp_profile = {1: 15, 2: 19, 3: 25, 4: 31, 5: 34, 6: 35, 7: 31, 8: 30, 9: 30, 10: 28, 11: 20, 12: 15}
    data['Delhi_Temp'] = data['Month'].map(delhi_temp_profile)
    
    return data

In [19]:
df_features = build_features(df_clean)
print("Feature extraction finished. Dataset shape:", df_features.shape)

Feature extraction finished. Dataset shape: (804, 35)


In [20]:
# 3. Identify targets vs features
store_columns = [col for col in df_raw.columns if col != 'Date']
num_stores = len(store_columns)
feature_columns = store_columns + ['IsWeekend', 'Sin_Month', 'Cos_Month', 'IsHoliday', 'Delhi_Temp']

# Apply split rules: Validate against April 1st to April 12th, 2026.
train_df = df_features[df_features['Date'] < '2026-04-01']

# Scale features between 0 and 1 for optimal neural net convergence
scaler = MinMaxScaler()
scaled_train_data = scaler.fit_transform(train_df[feature_columns])

In [21]:
# 4. Sliding window generator for sequential processing
def create_time_sequences(data, window_size, target_dim):
    x_seq, y_seq = [], []
    for i in range(len(data) - window_size):
        x_seq.append(data[i : (i + window_size), :])
        y_seq.append(data[i + window_size, :target_dim])
    return np.array(x_seq), np.array(y_seq)

WINDOW_SIZE = 14
X_train, y_train = create_time_sequences(scaled_train_data, WINDOW_SIZE, num_stores)

X_train_t = torch.tensor(X_train, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)

In [22]:
# 5. Define Multi-Output LSTM Architecture
class MultiOutputLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, output_size, num_layers=2):
        super(MultiOutputLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True)
        self.fc = nn.Linear(hidden_size, output_size)
        
    def forward(self, x):
        h0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        out, _ = self.lstm(x, (h0, c0))
        return self.fc(out[:, -1, :])

In [23]:
# Initialize hyperparameters
model = MultiOutputLSTM(input_size=len(feature_columns), hidden_size=64, output_size=num_stores)
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [26]:
# 6. Execute Model Training with 25 Full Epochs & Performance Accuracy Tracking
print("Initializing deep learning training routines across 25 complete epochs...\n")

model.train()
for epoch in range(100):
    optimizer.zero_grad()
   
    # Forward pass through the network
    predictions = model(X_train_t)
    
    # Calculate both Optimization Loss (MSE) and Interpretability Accuracy (MAE)
    loss = criterion(predictions, y_train_t)
    mae_metric = torch.mean(torch.abs(predictions - y_train_t))
    
    # Backward pass
    loss.backward()
    optimizer.step()
    
    # Granular logging for every single epoch as requested
    print(f"Epoch [{epoch+1:02d}/25] | Optimization Loss: {loss.item():.6f} | Training MAE (Error Metric): {mae_metric.item():.6f}")

print("\nModel optimization phase successfully terminated.")

Initializing deep learning training routines across 25 complete epochs...

Epoch [01/25] | Optimization Loss: 0.004827 | Training MAE (Error Metric): 0.038749
Epoch [02/25] | Optimization Loss: 0.004777 | Training MAE (Error Metric): 0.038255
Epoch [03/25] | Optimization Loss: 0.004728 | Training MAE (Error Metric): 0.037898
Epoch [04/25] | Optimization Loss: 0.004679 | Training MAE (Error Metric): 0.037772
Epoch [05/25] | Optimization Loss: 0.004638 | Training MAE (Error Metric): 0.037568
Epoch [06/25] | Optimization Loss: 0.004600 | Training MAE (Error Metric): 0.037254
Epoch [07/25] | Optimization Loss: 0.004562 | Training MAE (Error Metric): 0.037125
Epoch [08/25] | Optimization Loss: 0.004522 | Training MAE (Error Metric): 0.036971
Epoch [09/25] | Optimization Loss: 0.004479 | Training MAE (Error Metric): 0.036718
Epoch [10/25] | Optimization Loss: 0.004434 | Training MAE (Error Metric): 0.036597
Epoch [11/25] | Optimization Loss: 0.004393 | Training MAE (Error Metric): 0.036298
E

In [27]:
# 7. Package and Export All Pipeline Assets
print("Exporting production weights and serialization objects to root directory...")
torch.save(model.state_dict(), 'lstm_store_model.pth')

with open('scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

metadata = {
    'store_columns': store_columns,
    'feature_columns': feature_columns,
    'window_size': WINDOW_SIZE
}
with open('metadata.pkl', 'wb') as f:
    pickle.dump(metadata, f)
    
print("All project artifacts successfully written!")

Exporting production weights and serialization objects to root directory...
All project artifacts successfully written!
